# 🎙️ BAPS Voice Cloning - Basic Audio Generator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanmay0251/BAPS-Audio-Clone/blob/main/basic_audio_generator.ipynb)

Generate personalized Hindi audio messages for multiple names using voice cloning.

## Features
- ✓ Upload reference audio and clone the voice
- ✓ Upload Excel/CSV with names
- ✓ Generate personalized audio for each name
- ✓ Download all audio files as ZIP
- ✓ GPU accelerated (T4 recommended)

---

## 📦 Step 1: Setup & Installation

Install dependencies and import the utils module.

In [ ]:
# Create utils directory and download modules
!mkdir -p utils

# Download utils files from GitHub
!wget -q https://raw.githubusercontent.com/Tanmay0251/BAPS-Audio-Clone/main/utils/__init__.py -O utils/__init__.py
!wget -q https://raw.githubusercontent.com/Tanmay0251/BAPS-Audio-Clone/main/utils/voice_cloner_colab.py -O utils/voice_cloner_colab.py
!wget -q https://raw.githubusercontent.com/Tanmay0251/BAPS-Audio-Clone/main/utils/batch_generator.py -O utils/batch_generator.py
!wget -q https://raw.githubusercontent.com/Tanmay0251/BAPS-Audio-Clone/main/utils/audio_merger.py -O utils/audio_merger.py

print("✅ Utils downloaded!")

# Check GPU
!nvidia-smi

# Install dependencies
!pip install -q coqui-tts torch torchaudio pydub pandas openpyxl tqdm
!apt-get install -qq ffmpeg

print("\n✅ Setup complete!")

## 📤 Step 2: Upload Files

Upload:
1. **Reference audio** - MP4/WAV file with the voice to clone (6-15 seconds recommended)
2. **Names sheet** - Excel or CSV file with a column containing names

In [ ]:
from google.colab import files
import os

print("📤 Upload your reference audio file (MP4/WAV):")
uploaded = files.upload()
reference_audio = list(uploaded.keys())[0]
print(f"✅ Reference audio uploaded: {reference_audio}")

print("\n📤 Upload your names sheet (Excel/CSV):")
uploaded = files.upload()
names_sheet = list(uploaded.keys())[0]
print(f"✅ Names sheet uploaded: {names_sheet}")

## ⚙️ Step 3: Configuration

Set your template text and column name.

In [ ]:
# Your Hindi template with {name} placeholder
template_text = "नमस्ते {name}, आपका हार्दिक स्वागत है। BAPS परिवार की ओर से आशा है आप स्वस्थ और प्रसन्न हैं।"

# Column name in your sheet that contains the names
name_column = "Name"  # Change this if your column has a different name

# Language code
language = "hi"

print("✅ Configuration set!")
print(f"Template: {template_text}")
print(f"Name column: {name_column}")

## 🚀 Step 4: Initialize Voice Cloner

Load the model and clone the voice from reference audio.

In [ ]:
from utils.voice_cloner_colab import VoiceCloner
from utils.batch_generator import BatchAudioGenerator

# Initialize voice cloner
print("🎙️ Initializing voice cloner...")
voice_cloner = VoiceCloner(reference_audio, use_gpu=True)

# Test the voice
voice_cloner.test_voice("नमस्ते, यह एक परीक्षण है।")

print("\n✅ Voice cloner ready!")

## 👀 Step 5: Preview Names (Optional)

Check the first few names from your sheet.

In [ ]:
import pandas as pd

# Load sheet
if names_sheet.endswith('.csv'):
    df = pd.read_csv(names_sheet)
else:
    df = pd.read_excel(names_sheet)

print(f"📊 Total rows: {len(df)}")
print(f"📋 Columns: {', '.join(df.columns)}")
print(f"\n👀 First 5 names:")
display(df[name_column].head())

valid_names = df[name_column].dropna()
print(f"\n✅ Valid names to process: {len(valid_names)}")

## 🧪 Step 6: Test with Sample Names (Optional)

Generate audio for a few sample names to verify quality.

In [ ]:
# Initialize generator
generator = BatchAudioGenerator(
    voice_cloner=voice_cloner,
    template_text=template_text,
    output_dir="generated_audios"
)

# Test with first 3 names
test_names = df[name_column].dropna().head(3).tolist()
print(f"🧪 Testing with: {test_names}")

test_results = generator.generate_from_names(test_names, add_timestamp=False)

# Play first audio
if test_results['successful'] > 0:
    from IPython.display import Audio, display
    print("\n🔊 Playing first generated audio:")
    display(Audio(test_results['file_paths'][0]))

## 🎬 Step 7: Generate All Audio Files

Generate audio for all names in your sheet.

In [ ]:
# Generate audio for all names
results = generator.generate_from_sheet(
    sheet_path_or_df=df,
    name_column=name_column,
    add_timestamp=False
)

print("\n🎉 All audio files generated!")

## 📥 Step 8: Download Audio Files

Download all generated audio files as a ZIP archive.

In [ ]:
import shutil
from datetime import datetime

# Create ZIP
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"baps_audio_{timestamp}"

print("📦 Creating ZIP archive...")
shutil.make_archive(zip_filename, 'zip', 'generated_audios')

print(f"✅ ZIP created: {zip_filename}.zip")
print(f"📊 Total files: {results['successful']}")

# Download
from google.colab import files
files.download(f"{zip_filename}.zip")

print("\n🎉 Download complete!")

## 📊 Optional: View Results Summary

In [ ]:
import os

audio_files = [f for f in os.listdir('generated_audios') if f.endswith('.mp3')]

print("📋 Generated Audio Files:")
print("=" * 60)
for i, filename in enumerate(audio_files[:10], 1):
    file_size = os.path.getsize(f"generated_audios/{filename}") / 1024
    print(f"{i:3d}. {filename:40s} ({file_size:.1f} KB)")

if len(audio_files) > 10:
    print(f"\n... and {len(audio_files) - 10} more files")

total_size = sum(os.path.getsize(f"generated_audios/{f}") for f in audio_files) / (1024 * 1024)
print(f"\n📊 Total files: {len(audio_files)}")
print(f"💾 Total size: {total_size:.2f} MB")

---

## 🎉 Done!

### Tips:
- Use **GPU runtime** (Runtime → Change runtime type → T4 GPU)
- Reference audio: **6-15 seconds** of clear speech
- For 100-500 names, use the **optimized notebook**